# 04 - Cross-Product LGD Integration

**One Integrated LGD Framework - Australian Bank-Style Portfolio View**

This notebook standardises product comparisons across mortgage, commercial, development, CRE investment, residual stock, land/subdivision, bridging, and mezz/2nd mortgage.

| Step | Description |
|---|---|
| 1 | Build cross-product segmentation taxonomy |
| 2 | Assemble weighted LGD metrics by product |
| 3 | Compare weighted LGD, downturn sensitivity, and recovery time |
| 4 | Analyse portfolio mix and contribution |
| 5 | Create transparent risk ranking |


In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_all_datasets
from src.lgd_calculation import run_full_pipeline
from src.reproducibility import set_global_seed

set_global_seed(54)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

PROJECT_ROOT = Path('..').resolve()
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
TABLE_DIR = OUTPUT_DIR / 'tables'
FIGURE_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")

PRODUCT_ORDER = [
    'Residential Mortgage',
    'Commercial Cashflow',
    'Development Finance',
    'CRE Investment',
    'Residual Stock',
    'Land / Subdivision',
    'Bridging Loan',
    'Mezzanine / 2nd Mortgage',
]


## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** use observed workout inputs first, then approved proxy inputs, then conservative fallback with explicit disclosure.
- **Proxy logic (standard policy):** transparent proxy assumptions are permitted where observed workout tapes are unavailable.
- **Cure / resolution treatment:** cure is applied where product-appropriate (especially mortgage); recovery waterfall and collateral realisation logic are used for non-mortgage secured products.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status (standard policy):** this cross-product view is a portfolio-project proxy baseline, **not** internally calibrated to real workout outcomes.
- **Output standard:** compare `lgd_base`, `lgd_downturn`, `lgd_final`, downturn sensitivity, and base recovery-time metric across products using EAD-weighted definitions.


## Validation Context (Simulated)

This comparison is a **simulated validation view** using transparent proxy data and assumptions. It is designed for portfolio communication and interview discussion, not production model approval.


## 1. Cross-Product Segmentation Table


In [ ]:
segmentation_table = pd.DataFrame([
    {
        'product': 'Residential Mortgage',
        'repayment_source': 'Borrower income, cure, property sale, LMI where eligible',
        'security': 'First-ranking residential mortgage (with optional LMI support)',
        'drivers': 'LVR, LMI, cure rate, arrears proxy, foreclosure/workout path',
    },
    {
        'product': 'Commercial Cashflow',
        'repayment_source': 'Operating cashflow plus collateral/workout recoveries',
        'security': 'PPSR/GSA and other business security packages',
        'drivers': 'DSCR/ICR, leverage, security coverage, industry risk, workout strategy',
    },
    {
        'product': 'Development Finance',
        'repayment_source': 'Project completion and sales/refinance exit',
        'security': 'Project assets and assignment of project cashflows/presales',
        'drivers': 'GRV decline, completion %, cost-to-complete, sell-through delay, presales',
    },
    {
        'product': 'CRE Investment',
        'repayment_source': 'Rental income durability, refinance or forced sale',
        'security': 'Stabilised investment property security',
        'drivers': 'LVR, DSCR, WALE, vacancy, tenant concentration, cap-rate expansion',
    },
    {
        'product': 'Residual Stock',
        'repayment_source': 'Sale of completed but unsold units',
        'security': 'Residual completed stock inventory',
        'drivers': 'Unsold units, absorption speed, discount-to-clear, holding costs, time to sale',
    },
    {
        'product': 'Land / Subdivision',
        'repayment_source': 'Land lot disposal over extended sale horizon',
        'security': 'Land/subdivision collateral with low carry income',
        'drivers': 'Zoning/stage, liquidity risk, market depth, value haircut, time to sell',
    },
    {
        'product': 'Bridging Loan',
        'repayment_source': 'Planned refinance or sale exit',
        'security': 'Short-tenor property-backed bridge security',
        'drivers': 'Exit type, exit certainty, valuation risk, time to exit, failed-exit risk',
    },
    {
        'product': 'Mezzanine / 2nd Mortgage',
        'repayment_source': 'Residual collateral value after senior debt repayment',
        'security': 'Subordinated lien / second-ranking security',
        'drivers': 'Total LVR, attachment point, subordination, collateral value decline',
    },
])

display(segmentation_table)
segmentation_table.to_csv(TABLE_DIR / 'cross_product_segmentation_table.csv', index=False)


## 2. Build Integrated Product Comparison Inputs


In [ ]:
datasets = generate_all_datasets()
core_results = run_full_pipeline(datasets)


def _safe_weighted(df, value_col, ead_col):
    if value_col not in df.columns or ead_col not in df.columns:
        return np.nan
    x = pd.to_numeric(df[value_col], errors='coerce')
    w = pd.to_numeric(df[ead_col], errors='coerce').fillna(0.0)
    mask = x.notna() & w.notna()
    if mask.sum() == 0 or w.loc[mask].sum() <= 0:
        return np.nan
    return float((x.loc[mask] * w.loc[mask]).sum() / w.loc[mask].sum())


def _summary_from_core(product_label, key):
    df = core_results[key]['loans_with_overlays'].copy()
    base_col = 'lgd_base' if 'lgd_base' in df.columns else 'realised_lgd'
    downturn_col = 'lgd_downturn' if 'lgd_downturn' in df.columns else base_col
    final_col = 'lgd_final' if 'lgd_final' in df.columns else downturn_col
    recovery_col = 'workout_months' if 'workout_months' in df.columns else None

    return {
        'product': product_label,
        'loan_count': len(df),
        'total_ead': float(pd.to_numeric(df['ead'], errors='coerce').fillna(0.0).sum()),
        'ead_weighted_lgd_base': _safe_weighted(df, base_col, 'ead'),
        'ead_weighted_lgd_downturn': _safe_weighted(df, downturn_col, 'ead'),
        'ead_weighted_lgd_final': _safe_weighted(df, final_col, 'ead'),
        'mean_recovery_time_months': float(pd.to_numeric(df[recovery_col], errors='coerce').mean()) if recovery_col else np.nan,
        'data_source': 'core_pipeline',
        'coverage_status': 'available',
        'notes': 'explicit final LGD from core APRA overlay pipeline',
    }


def _summary_from_loan_csv(product_label, filename, ead_col, base_col, downturn_col, final_col, recovery_col):
    path = TABLE_DIR / filename
    if not path.exists():
        return {
            'product': product_label,
            'loan_count': 0,
            'total_ead': 0.0,
            'ead_weighted_lgd_base': np.nan,
            'ead_weighted_lgd_downturn': np.nan,
            'ead_weighted_lgd_final': np.nan,
            'mean_recovery_time_months': np.nan,
            'data_source': filename,
            'coverage_status': 'missing_output',
            'notes': 'run product notebook to generate output table',
        }

    df = pd.read_csv(path)
    required_cols = [ead_col, base_col, downturn_col, final_col, recovery_col]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        return {
            'product': product_label,
            'loan_count': int(len(df)),
            'total_ead': float(pd.to_numeric(df[ead_col], errors='coerce').fillna(0.0).sum()) if ead_col in df.columns else np.nan,
            'ead_weighted_lgd_base': np.nan,
            'ead_weighted_lgd_downturn': np.nan,
            'ead_weighted_lgd_final': np.nan,
            'mean_recovery_time_months': np.nan,
            'data_source': filename,
            'coverage_status': 'invalid_output',
            'notes': f"missing required columns: {', '.join(missing_cols)}",
        }

    return {
        'product': product_label,
        'loan_count': int(len(df)),
        'total_ead': float(pd.to_numeric(df[ead_col], errors='coerce').fillna(0.0).sum()),
        'ead_weighted_lgd_base': _safe_weighted(df, base_col, ead_col),
        'ead_weighted_lgd_downturn': _safe_weighted(df, downturn_col, ead_col),
        'ead_weighted_lgd_final': _safe_weighted(df, final_col, ead_col),
        'mean_recovery_time_months': float(pd.to_numeric(df[recovery_col], errors='coerce').mean()),
        'data_source': filename,
        'coverage_status': 'available',
        'notes': f'explicit final LGD from {final_col}; recovery time from {recovery_col}',
    }


rows = [
    _summary_from_core('Residential Mortgage', 'mortgage'),
    _summary_from_core('Commercial Cashflow', 'commercial'),
    _summary_from_core('Development Finance', 'development'),
    _summary_from_loan_csv(
        'CRE Investment',
        'cre_investment_loan_level_output.csv',
        ead_col='ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='time_to_recovery_months_base',
    ),
    _summary_from_loan_csv(
        'Residual Stock',
        'residual_stock_loan_level_output.csv',
        ead_col='ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='time_to_sale_base',
    ),
    _summary_from_loan_csv(
        'Land / Subdivision',
        'land_subdivision_loan_level_output.csv',
        ead_col='ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='time_to_sell_base',
    ),
    _summary_from_loan_csv(
        'Bridging Loan',
        'bridging_loan_level_output.csv',
        ead_col='ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='exit_time_base',
    ),
    _summary_from_loan_csv(
        'Mezzanine / 2nd Mortgage',
        'mezz_second_mortgage_loan_level_output.csv',
        ead_col='mezz_ead',
        base_col='lgd_base',
        downturn_col='lgd_downturn',
        final_col='lgd_final',
        recovery_col='time_to_recovery_base',
    ),
]

comparison_df = pd.DataFrame(rows)
comparison_df['downturn_sensitivity_pp'] = (
    comparison_df['ead_weighted_lgd_downturn'] - comparison_df['ead_weighted_lgd_base']
) * 100
comparison_df['downturn_sensitivity_ratio'] = (
    comparison_df['ead_weighted_lgd_downturn'] / comparison_df['ead_weighted_lgd_base']
)
comparison_df['final_minus_base_pp'] = (
    comparison_df['ead_weighted_lgd_final'] - comparison_df['ead_weighted_lgd_base']
) * 100

comparison_df['product'] = pd.Categorical(
    comparison_df['product'],
    categories=PRODUCT_ORDER,
    ordered=True,
)
comparison_df = comparison_df.sort_values('product').reset_index(drop=True)

print('Integrated product rows:', len(comparison_df))
print('Available rows:', int((comparison_df['coverage_status'] == 'available').sum()))
display(
    comparison_df[
        [
            'product',
            'loan_count',
            'total_ead',
            'ead_weighted_lgd_base',
            'ead_weighted_lgd_downturn',
            'ead_weighted_lgd_final',
            'downturn_sensitivity_pp',
            'mean_recovery_time_months',
            'coverage_status',
        ]
    ].round(4)
)


## 3. Compare Weighted LGD, Downturn Sensitivity, and Recovery Time


In [ ]:
available = comparison_df[comparison_df['coverage_status'] == 'available'].copy()

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

plot_lgd = available.melt(
    id_vars='product',
    value_vars=['ead_weighted_lgd_base', 'ead_weighted_lgd_downturn', 'ead_weighted_lgd_final'],
    var_name='metric',
    value_name='weighted_lgd',
)

sns.barplot(data=plot_lgd, x='product', y='weighted_lgd', hue='metric', ax=axes[0])
axes[0].set_title('Weighted LGD by Product (Base / Downturn / Final)')
axes[0].set_xlabel('Product')
axes[0].set_ylabel('EAD-weighted LGD')
axes[0].tick_params(axis='x', rotation=35)

sns.barplot(data=available, x='product', y='downturn_sensitivity_pp', color='#dd8452', ax=axes[1])
axes[1].set_title('Downturn Sensitivity by Product')
axes[1].set_xlabel('Product')
axes[1].set_ylabel('Downturn minus Base (percentage points)')
axes[1].tick_params(axis='x', rotation=35)

sns.barplot(data=available, x='product', y='mean_recovery_time_months', color='#4c72b0', ax=axes[2])
axes[2].set_title('Mean Recovery Time by Product')
axes[2].set_xlabel('Product')
axes[2].set_ylabel('Months')
axes[2].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig(FIGURE_DIR / 'cross_product_integrated_comparison.png', dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)
    print('Plot display skipped (set LGD_NOTEBOOK_SHOW_PLOTS=1 to show).')

comparison_view = available[
    [
        'product',
        'ead_weighted_lgd_base',
        'ead_weighted_lgd_downturn',
        'ead_weighted_lgd_final',
        'downturn_sensitivity_pp',
        'downturn_sensitivity_ratio',
        'mean_recovery_time_months',
        'notes',
    ]
].copy()

comparison_view = comparison_view.sort_values('ead_weighted_lgd_downturn', ascending=False).reset_index(drop=True)
print('Cross-product weighted comparison:')
display(comparison_view.round(4))


## 4. Portfolio Mix Analysis


In [ ]:
mix_df = available.copy()
total_portfolio_ead = mix_df['total_ead'].sum()

mix_df['portfolio_ead_share'] = mix_df['total_ead'] / total_portfolio_ead
mix_df['portfolio_base_lgd_contribution'] = mix_df['portfolio_ead_share'] * mix_df['ead_weighted_lgd_base']
mix_df['portfolio_downturn_lgd_contribution'] = mix_df['portfolio_ead_share'] * mix_df['ead_weighted_lgd_downturn']
mix_df['portfolio_final_lgd_contribution'] = mix_df['portfolio_ead_share'] * mix_df['ead_weighted_lgd_final']

portfolio_summary = pd.DataFrame([
    {
        'portfolio_total_ead': total_portfolio_ead,
        'portfolio_weighted_lgd_base': mix_df['portfolio_base_lgd_contribution'].sum(),
        'portfolio_weighted_lgd_downturn': mix_df['portfolio_downturn_lgd_contribution'].sum(),
        'portfolio_weighted_lgd_final': mix_df['portfolio_final_lgd_contribution'].sum(),
        'products_with_available_metrics': len(mix_df),
    }
])

print('Portfolio integrated summary:')
display(portfolio_summary.round(6))

mix_view = mix_df[
    [
        'product',
        'total_ead',
        'portfolio_ead_share',
        'ead_weighted_lgd_final',
        'portfolio_final_lgd_contribution',
    ]
].sort_values('portfolio_ead_share', ascending=False).reset_index(drop=True)

display(mix_view.round(4))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(mix_view['product'], mix_view['portfolio_ead_share'], color='#55a868')
ax.invert_yaxis()
ax.set_title('Portfolio Mix by Product (EAD Share)')
ax.set_xlabel('Share of Portfolio EAD')
ax.set_ylabel('Product')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'cross_product_portfolio_mix.png', dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)


## 5. Cross-Product Risk Ranking


In [ ]:
risk_df = available.copy()

risk_df['rank_lgd_level'] = risk_df['ead_weighted_lgd_downturn'].rank(ascending=False, method='dense')
risk_df['rank_downturn_sensitivity'] = risk_df['downturn_sensitivity_pp'].rank(ascending=False, method='dense')
risk_df['rank_recovery_time'] = risk_df['mean_recovery_time_months'].rank(ascending=False, method='dense')

rank_cols = ['rank_lgd_level', 'rank_downturn_sensitivity', 'rank_recovery_time']
risk_df['composite_risk_score'] = risk_df[rank_cols].mean(axis=1, skipna=True)
risk_df['risk_rank'] = risk_df['composite_risk_score'].rank(ascending=True, method='dense')

risk_ranking = risk_df[
    [
        'risk_rank',
        'product',
        'ead_weighted_lgd_downturn',
        'downturn_sensitivity_pp',
        'mean_recovery_time_months',
        'composite_risk_score',
        'rank_lgd_level',
        'rank_downturn_sensitivity',
        'rank_recovery_time',
    ]
].sort_values(['risk_rank', 'ead_weighted_lgd_downturn'], ascending=[True, False]).reset_index(drop=True)

print('Risk ranking (higher rank = higher relative risk in this proxy framework):')
display(risk_ranking.round(4))


## Assumptions and Limitations

- This integrated comparison uses synthetic data and transparent proxy assumptions.
- Non-core secured products now export explicit `lgd_final` outputs (rather than using downturn as a proxy) for more apples-to-apples comparison.
- CRE now exports explicit `time_to_recovery_months_base`, defined as expected months from default to economic resolution.
- Out-of-time and vintage validation in this repo is simulated and should be recalibrated with real internal workout datasets before production use.


In [ ]:
# Export cross-product outputs
comparison_df.to_csv(TABLE_DIR / 'cross_product_comparison.csv', index=False)
mix_view.to_csv(TABLE_DIR / 'cross_product_portfolio_mix.csv', index=False)
risk_ranking.to_csv(TABLE_DIR / 'cross_product_risk_ranking.csv', index=False)
portfolio_summary.to_csv(TABLE_DIR / 'cross_product_portfolio_summary.csv', index=False)

print('Saved cross-product outputs:')
print('- cross_product_segmentation_table.csv')
print('- cross_product_comparison.csv')
print('- cross_product_portfolio_mix.csv')
print('- cross_product_risk_ranking.csv')
print('- cross_product_portfolio_summary.csv')
